Import Libraries

In [ ]:
from ultralytics import YOLO

In [ ]:
import pandas as pd
import numpy as np
import cv2, os, math, pickle

Load segmentation models

In [ ]:
scale_model = YOLO(r"D:\sardine scale MLmodel\v1\runs\segment\train\weights\best.pt")  
fish_model = YOLO(r"C:\Users\sowmy\Downloads\fish_freshness_classification - Copy\saved_models\segmentation\kartik\fish.pt")  

In [ ]:
def circumference_area(img):
    # Load the segmented fish image
    # image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Threshold the image to create a binary mask
    _, binary_mask = cv2.threshold(image, 1, 255, cv2.THRESH_BINARY)

    # Find contours in the binary mask
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Ensure at least one contour was found
    if len(contours) > 0:
        # Get the largest contour (presumably the fish)
        largest_contour = max(contours, key=cv2.contourArea)
        
        # Calculate the circumference of the contour
        circumference = cv2.arcLength(largest_contour, closed=True)
        
        # Calculate the area enclosed by the contour
        area = cv2.contourArea(largest_contour)
        
        return circumference, area

In [ ]:
def get_segmented_img(pred, index=0):
    img_shape = pred.orig_shape
    img = pred.orig_img.copy()
    height, width = img_shape[0], img_shape[1]
    masks = pred.masks.xy
    seg_img_list = []
    for i in range(len(masks)):
        if i != index: continue
        mask = masks[i]
        mask_points = mask.astype(int)
        binary_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(binary_mask, [mask_points], 255)
        masked_img = cv2.bitwise_and(img, img, mask=binary_mask)
        x, y, w, h = cv2.boundingRect(mask_points)
        segment_image = np.zeros((height, width, 3), dtype=np.uint8)
        segment_image[y:y+h, x:x+w] = masked_img[y:y+h, x:x+w]
        segment_image = segment_image[y:y+h, x:x+w] 
        white_background = False
        if white_background:
            black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
            segment_image[black_mask] = [255, 255, 255]
        seg_img_list.append(segment_image)
    return seg_img_list

In [ ]:
def get_object_height(img_path, model, conf=0.5, iou=0.7, seg=False):
    seg_img_list = None
    results = model(img_path, conf=conf, iou=iou, verbose=False)
    if len(results[0].boxes.cls) != 0:
        index = 0
        hp = int(results[0].boxes.xywh[index][3])
        wp = int(results[0].boxes.xywh[index][2])
        dp = int(math.hypot(hp, wp))
        if hp<wp: (hp, wp) = (wp, hp)
        obj_present = True
        if seg: seg_img_list = get_segmented_img(results[0], index)
    else:
        hp, wp, dp = (0, 0, 0)
        obj_present = False
    return hp, wp, dp, obj_present, seg_img_list


def fish_measures(img_path):
    # Predict with the scale model
    scale_hp, scale_wp, scale_dp, scale, _ = get_object_height(img_path, scale_model)
    # Predict with the fish model
    fish_hp, fish_wp, fish_dp, fish, fish_img_list = get_object_height(img_path, fish_model, 
                                                     conf=0.75, iou=0.7, seg=True)
    
    try:
        scale_hcm = 16
        one_pixel_value = scale_hcm / scale_hp
        fish_hcm = int(fish_hp * one_pixel_value)
        fish_wcm = int(fish_wp * one_pixel_value)
        fish_dcm = int(fish_dp * one_pixel_value)
        scale_dcm = int(scale_dp * one_pixel_value)
        scale_wcm = int(scale_wp * one_pixel_value)
    except:
        (fish_hcm, fish_wcm, fish_dcm, 
         scale_dcm, scale_wcm, one_pixel_value) = (0, 0, 0, 0, 0, 0)
        
    return (fish_hcm, fish_wcm, fish_dcm, scale_dcm, scale_wcm, 
            fish, scale, fish_img_list, one_pixel_value)

In [ ]:
def single_img_data(img_path):
    filename = os.path.basename(img_path)
    (fish_hcm, fish_wcm, fish_dcm, scale_dcm, scale_wcm, fish, scale, 
     fish_img_list, one_pixel_value) = fish_measures(img_path)
    data = []
    if fish_img_list != None:
        for fish_img in fish_img_list:
            circumference, area = circumference_area(fish_img)
            if one_pixel_value != 0:
                circumference = int(circumference * one_pixel_value)
                area = int(area * one_pixel_value * one_pixel_value)
            data.append((filename, fish_hcm, fish_wcm, fish_dcm, 
                         scale_dcm, scale_wcm, one_pixel_value,
                         scale, fish, circumference, area))
    else:
        data.append((filename, fish_hcm, fish_wcm, fish_dcm, 
                     scale_dcm, scale_wcm, one_pixel_value,
                     scale, fish, 0, 0))
    return data

In [ ]:
img_path = r"D:\sardine scale MLmodel\v3\test\Good\20240224091224451_sardine_good.jpeg"
data = single_img_data(img_path)

In [ ]:
len(data)

1

In [ ]:
# Create a DataFrame
df = pd.DataFrame(data, columns=['Filename', 'FishHeight', 'FishWidth', 
                                 'FishDiagonal', 'ScaleDiagonal', 'ScaleWidth',
                                 'OnePixelValue', 'Scale', 'Fish', 
                                 'Circumference', 'Area'])


,Filename,FishHeight,FishWidth,FishDiagonal,ScaleDiagonal,ScaleWidth,OnePixelValue,Scale,Fish,Circumference,Area
0,20240224091224451_sardine_good.jpeg,16,3,17,16,1,0.008197,True,True,41,34


Load the ML model

In [ ]:
model_path = r"D:\sardine scale MLmodel\v3\models\Logistic Regression.pkl"

In [ ]:
with open(model_path, 'rb') as f:
    model = pickle.load(f)

In [ ]:
def pred_single_img(test, model):
    probabilities = []
    for index, row in test.iterrows():
        if row['Scale']==False:
            print(f'Scale not detected @ {index}')
            break # go to softness model
        if row['Fish']==False:
            print(f'Fish not detected @ {index}')
            break
        Height = row['FishHeight']
        Width = row['FishWidth']
        diagonal = row['FishDiagonal']
        Circumference = row['Circumference']
        Area = row['Area']
        X_test = [[Height, Width, diagonal, Circumference, Area]]
        # pred = model.predict(X_test)
        # Get probability estimates for the predicted labels
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)
            # print(f"Probability :", y_proba)
        probabilities.append(y_proba)
    return probabilities

In [ ]:
probabilities = pred_single_img(df, model)

Probability : [[0.59628599 0.40371401]]


c:\Users\sowmy\anaconda3\envs\opencv_env\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\sowmy\anaconda3\envs\opencv_env\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [ ]:
# Set your chosen threshold
threshold = 0.8  

# Adjust predictions based on the threshold
predictions = np.ones(len(probabilities))
for i in range(len(probabilities)):
    if probabilities[i][0][0] >= threshold:
        predictions[i] = 0

In [ ]:
label_map = {0: 'Bad', 1: 'Good'}
for pred in predictions :
    print(label_map[int(pred)])

In [ ]:
import matplotlib.pyplot as plt

img = cv2.imread(img_path)
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(rgb);
title = label_map[int(predictions[0])]
plt.title('predicted: '+title);